# Global Gold Price Intelligence & Historical Exploration
## Research-Grade Historical Data Engineering Pipeline (Task 01)

### Objectives:
1. **20+ Year Global Price History**: COMEX continuous gold futures (`GC=F`) in USD per troy ounce (2000–2026).
2. **Indian Domestic Bullion**: Official Reserve Bank of India (RBI) Mumbai benchmark spot prices (INR per 10 grams) & calculated domestic parity.
3. **Foreign Exchange Rates**: USD/INR historical exchange rates (2003–2026).
4. **Macroeconomic Regimes**: Quantitative inspection of market dynamics around the **2008 Global Financial Crisis**, **2020 COVID-19 Pandemic**, and **2024–2026 Structural Bull Regime**.
5. **Data Quality Integrity**: Strict verification that no prices were synthetically interpolated or fabricated.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine

# Set display aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
print('Data engineering packages loaded successfully!')

## 1. Load Processed Research Datasets

In [ ]:
# Load master daily cross-market dataset
df_master = pd.read_csv('../data/processed/gold_master_daily_unified.csv')
df_master['date'] = pd.to_datetime(df_master['date'])

# Load RBI benchmark series
df_rbi_annual = pd.read_csv('../data/processed/gold_india_rbi_annual.csv')
df_rbi_monthly = pd.read_csv('../data/processed/gold_india_rbi_monthly.csv')

# Load regime performance analytics
df_regimes = pd.read_csv('../data/processed/regime_performance_metrics.csv')

print(f"Master Daily Records: {len(df_master):,} rows spanning {df_master['date'].min().date()} to {df_master['date'].max().date()}")
print(f"RBI Annual Records: {len(df_rbi_annual)} rows spanning FY {df_rbi_annual['financial_year'].min()} to {df_rbi_annual['financial_year'].max()}")
df_master.head()

## 2. 26-Year Historical Trajectory (USD per Troy Ounce)
Visualize continuous COMEX Gold Futures prices from August 2000 through late 2026.

In [ ]:
gold_valid = df_master[df_master['gold_market_open']].copy()

fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(gold_valid['date'], gold_valid['close_usd_per_troy_oz'], label='COMEX Gold Futures (USD/ozt)', color='#D4AF37', linewidth=1.5)

# Annotate key milestones
ax.axvspan(pd.to_datetime('2007-12-01'), pd.to_datetime('2009-06-30'), color='#ff9999', alpha=0.3, label='2008 GFC')
ax.axvspan(pd.to_datetime('2020-01-01'), pd.to_datetime('2021-12-31'), color='#99ccff', alpha=0.3, label='2020 COVID-19')
ax.axvspan(pd.to_datetime('2024-01-01'), pd.to_datetime('2026-12-31'), color='#ffcc99', alpha=0.3, label='2024-2026 Bull Run')

ax.set_title('Global Gold Price Evolution: 2000 – 2026 (USD per Troy Ounce)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Date', fontweight='bold')
ax.set_ylabel('Price (USD / Troy Ounce)', fontweight='bold')
ax.legend(loc='upper left', frameon=True)
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 3. Macroeconomic Regime Analysis (2008 GFC, 2020 COVID, 2024–2026 Record Highs)
Detailed inspection of risk, return, and volatility across identified market eras.

In [ ]:
display(df_regimes[['regime_name', 'observations', 'start_price_usd', 'end_price_usd', 'regime_return_pct', 'annualized_volatility_pct', 'max_drawdown_pct']])

In [ ]:
# Visualizing annualized volatility & max drawdown across regimes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Volatility bar chart
colors = ['#4682B4' if not 'Crisis' in r and not 'Shock' in r and not 'Bull' in r else '#E24A4A' for r in df_regimes['regime_name']]
ax1.barh(df_regimes['regime_name'], df_regimes['annualized_volatility_pct'], color=colors)
ax1.set_title('Annualized Volatility (% p.a.) by Market Regime', fontweight='bold')
ax1.set_xlabel('Volatility (%)')
ax1.grid(True, linestyle='--', alpha=0.6)

# Regime Return bar chart
ret_colors = ['#2CA02C' if val >= 0 else '#D62728' for val in df_regimes['regime_return_pct']]
ax2.barh(df_regimes['regime_name'], df_regimes['regime_return_pct'], color=ret_colors)
ax2.set_title('Total Cumulative Return (%) by Market Regime', fontweight='bold')
ax2.set_xlabel('Return (%)')
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 4. Indian Gold Valuation: Parity Dynamics & USD/INR Exchange Rate Impact
Domestic gold prices in India are driven by two coupled engines:
1. **International USD Gold Price** ($/ozt)
2. **USD/INR Exchange Rate** (₹/$ depreciation)

$$\text{Gold Parity (INR per 10g)} = \left( \frac{\text{Price}_{\text{USD/ozt}}}{31.1034768} \right) \times 10 \times \text{USD/INR}$$

In [ ]:
df_parity = df_master[df_master['both_markets_open']].copy()

fig, ax1 = plt.subplots(figsize=(15, 6))
color = '#D4AF37'
ax1.set_xlabel('Date', fontweight='bold')
ax1.set_ylabel('Theoretical Domestic Parity (INR / 10g)', color=color, fontweight='bold')
ax1.plot(df_parity['date'], df_parity['gold_parity_inr_per_10g'], color=color, linewidth=1.5, label='Gold Domestic Parity (INR/10g)')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = '#2B5B84'
ax2.set_ylabel('USD / INR Exchange Rate', color=color, fontweight='bold')
ax2.plot(df_parity['date'], df_parity['fx_usdinr'], color=color, linewidth=1.2, linestyle='--', label='USD/INR FX')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Indian Domestic Gold Parity (INR / 10g) vs USD/INR Exchange Rate', fontsize=14, fontweight='bold', pad=12)
fig.tight_layout()
plt.show()

## 5. Official RBI Domestic Spot Price (Mumbai) vs International London Price
Comparison of 40-year official RBI benchmark data (Table 35) showing the structural import tariff and domestic spread in India.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
x_labels = df_rbi_annual['financial_year']
ax.plot(x_labels, df_rbi_annual['gold_mumbai_inr_per_10g'], marker='o', color='#B8860B', label='RBI Mumbai Spot (₹/10g)', linewidth=2)
ax.set_title('RBI Handbook: 40-Year Official Mumbai Spot Gold Price (1985-86 to 2025-26)', fontsize=14, fontweight='bold')
ax.set_xlabel('Financial Year', fontweight='bold')
ax.set_ylabel('INR per 10 grams', fontweight='bold')
ax.set_xticks(range(0, len(x_labels), 3))
ax.set_xticklabels(x_labels.iloc[::3], rotation=45)
ax.legend(loc='upper left')
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 6. Data Integrity & Non-Interpolation Verification
Auditing the missing values policy: confirming that holidays and exchange closures are left authentic rather than filled with synthetic interpolated prices.

In [ ]:
# Verify zero interpolation
any_interpolated = df_master['is_interpolated'].any()
print(f"Any synthetic/interpolated rows: {any_interpolated} (Expected: False)")

# Market trading coverage summary
total_rows = len(df_master)
gold_open = df_master['gold_market_open'].sum()
fx_open = df_master['fx_market_open'].sum()
both_open = df_master['both_markets_open'].sum()

print(f"Total Calendar Records: {total_rows:,}")
print(f"Gold Market Trading Days: {gold_open:,} ({gold_open/total_rows*100:.1f}%)")
print(f"FX Market Trading Days: {fx_open:,} ({fx_open/total_rows*100:.1f}%)")
print(f"Joint Trading Days: {both_open:,} ({both_open/total_rows*100:.1f}%)")

## 7. Database Querying via SQLAlchemy
Validating that SQLite / PostgreSQL tables can be seamlessly queried for downstream modeling.

In [ ]:
engine = create_engine('sqlite:///../data/gold_intelligence.db')
df_db_check = pd.read_sql("""
    SELECT regime_name, 
           COUNT(*) as trading_days,
           ROUND(AVG(close_usd_per_troy_oz), 2) as avg_gold_usd,
           ROUND(AVG(fx_usdinr), 2) as avg_fx_usdinr,
           ROUND(AVG(gold_parity_inr_per_10g), 2) as avg_parity_inr_10g
    FROM gold_daily_prices
    WHERE gold_market_open = 1 AND fx_market_open = 1
    GROUP BY regime_name
    ORDER BY MIN(date)
""", engine)

display(df_db_check)